In [ ]:
# =====================================================================
#  Legal-J : InLegalBERT fine-tuning on ILDC  (version 3)
#
#  What changed compared with version 2
#   1. Training data: ildc_multi_train.csv is added to
#      ildc_single_train.csv (several times more judgments). Any
#      judgment that also appears in the dev or test files is removed
#      from training, so the scores stay honest.
#   2. Dev set: ildc_single_dev + ildc_multi_dev (minus anything in test).
#   3. Up to 8 windows per judgment are available (was 4). Training
#      uses 4 of them per judgment each epoch (the last 2 always, 2
#      more picked at random), prediction can average up to 8.
#   4. The model is checked on dev twice per epoch, because one epoch is
#      now long; the best check is kept.
# =====================================================================
import os
import json
import random
import time
import hashlib

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from transformers.utils import logging as hf_logging
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)
from tqdm import tqdm

# ------------------------------- settings ----------------------------
DATA_DIR = "/kaggle/input/datasets/dataset"
TRAIN_FILES = ["ildc_single_train.csv", "ildc_multi_train.csv"]
DEV_FILES = ["ildc_single_dev.csv", "ildc_multi_dev.csv"]
TEST_FILE = "ildc_test.csv"

MODEL_NAME = "law-ai/InLegalBERT"
OUTPUT_DIR = "mvp_inlegalbert_best"

MAX_LEN = 512
CONTENT = MAX_LEN - 2      # tokens per window, without [CLS] and [SEP]
STRIDE = 384               # step between windows (126 tokens overlap)
MAX_WINDOWS = 8            # windows per judgment available, counted from the end
TRAIN_WINDOWS = 4          # windows per judgment used in each training epoch
ALWAYS_LAST = 2            # of those, the last 2 windows are always used

BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
EPOCHS = 3
EVALS_PER_EPOCH = 2
LR = 2e-5
LAYER_DECAY = 0.95         # lower layers get a smaller learning rate
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
PATIENCE = 2               # dev checks without improvement before stopping
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ------------------------------- data --------------------------------
def load_csv(name):
    path = os.path.join(DATA_DIR, name)
    df = pd.read_csv(path)
    missing = {"text", "label"} - set(df.columns)
    if missing:
        raise ValueError(f"{name} has columns {list(df.columns)} but needs 'text' and 'label'")
    df = df[["text", "label"]].dropna(subset=["label"]).copy()
    df["text"] = df["text"].fillna("").astype(str)
    df["label"] = df["label"].astype(int)
    # key = fingerprint of the text with whitespace normalised, to find duplicates
    df["key"] = df["text"].map(lambda t: hashlib.md5(" ".join(t.split()).encode("utf-8")).hexdigest())
    return df


def load_split(names):
    frames = []
    for name in names:
        if os.path.exists(os.path.join(DATA_DIR, name)):
            df = load_csv(name)
            print(f"   {name}: {len(df)} rows")
            frames.append(df)
        else:
            print(f"   {name}: not found, skipped")
    return pd.concat(frames, ignore_index=True).drop_duplicates(subset="key").reset_index(drop=True)


def load_data():
    print("loading data")
    test_df = load_split([TEST_FILE])
    dev_df = load_split(DEV_FILES)
    train_df = load_split(TRAIN_FILES)

    test_keys = set(test_df["key"])
    in_test = dev_df["key"].isin(test_keys)
    dev_df = dev_df[~in_test].reset_index(drop=True)
    dev_keys = set(dev_df["key"])

    train_in_test = train_df["key"].isin(test_keys)
    train_in_dev = train_df["key"].isin(dev_keys)
    train_df = train_df[~(train_in_test | train_in_dev)].reset_index(drop=True)
    print(
        f"removed overlaps: dev judgments also in test {int(in_test.sum())} | "
        f"train judgments also in test {int(train_in_test.sum())} | "
        f"train judgments also in dev {int(train_in_dev.sum())}"
    )
    return train_df, dev_df, test_df


def tokenize_all(texts, tokenizer, desc):
    """Tokenise every document once and keep the ids in memory."""
    out = []
    for i in tqdm(range(0, len(texts), 64), desc=desc, mininterval=30):
        enc = tokenizer(
            texts[i:i + 64],
            add_special_tokens=False,
            truncation=False,
            return_attention_mask=False,
            return_token_type_ids=False,
        )
        out.extend(np.asarray(ids, dtype=np.int32) for ids in enc["input_ids"])
    return out


def make_windows(n_tokens):
    """(start, end) token windows, anchored at the END of the document.
    Index 0 is always the last CONTENT tokens."""
    if n_tokens <= CONTENT:
        return [(0, n_tokens)]
    windows = []
    end = n_tokens
    while len(windows) < MAX_WINDOWS:
        if end <= CONTENT:
            # reached the start: add a head window only if it adds new text
            if windows[-1][0] >= 64:
                windows.append((0, CONTENT))
            break
        windows.append((end - CONTENT, end))
        end -= STRIDE
    return windows


class WindowDataset(Dataset):
    """One item = one window of one judgment.
    per_doc=None  -> every window is used (dev / test).
    per_doc=N     -> N windows per judgment, re-drawn by resample() (train)."""

    def __init__(self, doc_ids, labels, tokenizer, per_doc=None):
        self.doc_ids = doc_ids
        self.labels = labels
        self.cls_id = tokenizer.cls_token_id
        self.sep_id = tokenizer.sep_token_id
        self.pad_id = tokenizer.pad_token_id
        self.per_doc = per_doc
        self.windows = [make_windows(len(ids)) for ids in doc_ids]
        self.rng = np.random.default_rng(SEED)
        self.items = []  # (document index, window rank, start, end)
        self.resample()

    def resample(self):
        items = []
        for d, wins in enumerate(self.windows):
            n = len(wins)
            if self.per_doc is not None and n > self.per_doc:
                fixed = list(range(ALWAYS_LAST))
                extra = self.rng.choice(np.arange(ALWAYS_LAST, n), size=self.per_doc - ALWAYS_LAST, replace=False)
                ranks = fixed + sorted(int(r) for r in extra)
            else:
                ranks = range(n)
            for r in ranks:
                s, e = wins[r]
                items.append((d, r, s, e))
        self.items = items

    def __len__(self):
        return len(self.items)

    def window_labels(self):
        return np.array([self.labels[d] for d, _, _, _ in self.items])

    def __getitem__(self, idx):
        d, rank, s, e = self.items[idx]
        ids = torch.from_numpy(self.doc_ids[d][s:e].astype(np.int64))
        n = len(ids)
        input_ids = torch.full((MAX_LEN,), self.pad_id, dtype=torch.long)
        attention_mask = torch.zeros(MAX_LEN, dtype=torch.long)
        input_ids[0] = self.cls_id
        input_ids[1:1 + n] = ids
        input_ids[1 + n] = self.sep_id
        attention_mask[:n + 2] = 1
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": torch.tensor(self.labels[d], dtype=torch.long),
            "doc": torch.tensor(d, dtype=torch.long),
            "rank": torch.tensor(rank, dtype=torch.long),
        }


# ---------------------------- evaluation -----------------------------
@torch.no_grad()
def predict_windows(model, loader):
    """Returns, for every window: (logit_accepted - logit_rejected),
    its document index and its rank (0 = last window of the document)."""
    model.eval()
    diffs, docs, ranks = [], [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        logits = logits.float()
        diffs.append((logits[:, 1] - logits[:, 0]).cpu().numpy())
        docs.append(batch["doc"].numpy())
        ranks.append(batch["rank"].numpy())
    return np.concatenate(diffs), np.concatenate(docs), np.concatenate(ranks)


def doc_probs(diffs, docs, ranks, n_docs, k):
    """P(accepted) per document = sigmoid(mean score of its last k windows)."""
    keep = ranks < k
    total = np.bincount(docs[keep], weights=diffs[keep], minlength=n_docs)
    count = np.bincount(docs[keep], minlength=n_docs)
    mean = total / np.maximum(count, 1)
    return 1.0 / (1.0 + np.exp(-mean))


def best_threshold(probs, labels):
    grid = np.round(np.linspace(0.05, 0.95, 91), 2)
    accs = np.array([accuracy_score(labels, (probs >= t).astype(int)) for t in grid])
    best = accs.max()
    candidates = grid[accs >= best - 1e-12]
    thr = float(candidates[np.argmin(np.abs(candidates - 0.5))])
    return thr, float(best)


def evaluate_dev(model, loader, labels):
    """Tries averaging the last 1..MAX_WINDOWS windows; for each, finds the
    threshold with the best dev accuracy. Returns the best combination."""
    diffs, docs, ranks = predict_windows(model, loader)
    labels = np.asarray(labels)
    rows = []
    for k in range(1, MAX_WINDOWS + 1):
        probs = doc_probs(diffs, docs, ranks, len(labels), k)
        thr, tuned_acc = best_threshold(probs, labels)
        rows.append({
            "k": k,
            "acc@0.5": accuracy_score(labels, (probs >= 0.5).astype(int)),
            "threshold": thr,
            "acc@thr": tuned_acc,
            "macro_f1@thr": f1_score(labels, (probs >= thr).astype(int), average="macro"),
            "auc": roc_auc_score(labels, probs),
        })
    best = max(rows, key=lambda r: (r["acc@thr"], r["auc"]))
    return best, rows


# ----------------------------- optimiser -----------------------------
def build_optimizer(model):
    """AdamW with layer-wise learning-rate decay; no weight decay on
    biases and LayerNorm parameters."""
    no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias")
    base = model.base_model
    layers = [base.embeddings] + list(base.encoder.layer)
    groups, seen = [], set()

    def add(named_params, lr):
        decay_params, plain_params = [], []
        for name, p in named_params:
            if not p.requires_grad or id(p) in seen:
                continue
            seen.add(id(p))
            (plain_params if any(nd in name for nd in no_decay) else decay_params).append(p)
        if decay_params:
            groups.append({"params": decay_params, "lr": lr, "weight_decay": WEIGHT_DECAY})
        if plain_params:
            groups.append({"params": plain_params, "lr": lr, "weight_decay": 0.0})

    n = len(layers)
    for i, layer in enumerate(layers):
        add(layer.named_parameters(), LR * (LAYER_DECAY ** (n - i)))
    add(model.named_parameters(), LR)  # pooler + classifier head
    return torch.optim.AdamW(groups, lr=LR)


# ------------------------------- train -------------------------------
def describe(name, df, doc_ids, ds):
    lens = np.array([len(x) for x in doc_ids])
    print(
        f"{name:5s} docs {len(df):6d} | accepted {df['label'].mean() * 100:5.1f}% | "
        f"tokens median {int(np.median(lens)):5d}, mean {int(lens.mean()):5d} | "
        f"windows per pass {len(ds)}"
    )


def train():
    set_seed(SEED)
    hf_logging.set_verbosity_error()
    print(f"device: {DEVICE} | mixed precision: {USE_AMP}")

    train_df, dev_df, test_df = load_data()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

    train_ids = tokenize_all(train_df["text"].tolist(), tokenizer, "tokenising train")
    dev_ids = tokenize_all(dev_df["text"].tolist(), tokenizer, "tokenising dev")
    test_ids = tokenize_all(test_df["text"].tolist(), tokenizer, "tokenising test")

    train_ds = WindowDataset(train_ids, train_df["label"].tolist(), tokenizer, per_doc=TRAIN_WINDOWS)
    dev_ds = WindowDataset(dev_ids, dev_df["label"].tolist(), tokenizer)
    test_ds = WindowDataset(test_ids, test_df["label"].tolist(), tokenizer)
    describe("train", train_df, train_ids, train_ds)
    describe("dev", dev_df, dev_ids, dev_ds)
    describe("test", test_df, test_ids, test_ds)

    workers = 2 if DEVICE.type == "cuda" else 0
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=workers, pin_memory=USE_AMP, drop_last=True)
    dev_loader = DataLoader(dev_ds, batch_size=EVAL_BATCH_SIZE, num_workers=workers, pin_memory=USE_AMP)
    test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, num_workers=workers, pin_memory=USE_AMP)

    # fully class-balanced loss (computed on the windows actually trained on)
    counts = np.bincount(train_ds.window_labels(), minlength=2).astype(np.float64)
    class_weights = torch.tensor(counts.sum() / (2.0 * counts), dtype=torch.float32).to(DEVICE)
    print("class weights [rejected, accepted]:", [round(w, 3) for w in class_weights.tolist()])
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = build_optimizer(model)
    steps_per_epoch = len(train_loader)
    total_steps = steps_per_epoch * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * WARMUP_RATIO),
        num_training_steps=total_steps,
    )
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    eval_steps = {max(1, round(steps_per_epoch * (i + 1) / EVALS_PER_EPOCH)) for i in range(EVALS_PER_EPOCH)}
    print(f"steps per epoch {steps_per_epoch} | dev check after steps {sorted(eval_steps)} of each epoch")

    best = None
    patience_counter = 0
    stop = False
    dev_labels = dev_df["label"].tolist()
    run_start = time.time()

    for epoch in range(EPOCHS):
        if epoch > 0:
            train_ds.resample()  # draw a fresh set of windows for this epoch
        model.train()
        running_loss, running_n = 0.0, 0
        for step, batch in enumerate(tqdm(train_loader, desc=f"epoch {epoch + 1}", mininterval=120), start=1):
            input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss = criterion(logits.float(), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item()
            running_n += 1

            if step not in eval_steps:
                continue

            # ---------------------- dev check ------------------------
            result, rows = evaluate_dev(model, dev_loader, dev_labels)
            model.train()
            point = epoch + step / steps_per_epoch
            minutes = (time.time() - run_start) / 60
            print(f"\nepoch {point:.2f} | train_loss {running_loss / running_n:.4f} | {minutes:.0f} min since start")
            running_loss, running_n = 0.0, 0
            for r in rows:
                print(
                    f"   dev, last {r['k']} window(s): acc@0.5 {r['acc@0.5']:.4f} | "
                    f"best threshold {r['threshold']:.2f} -> acc {r['acc@thr']:.4f} | "
                    f"macro-F1 {r['macro_f1@thr']:.4f} | AUC {r['auc']:.4f}"
                )

            if best is None or (result["acc@thr"], result["auc"]) > (best["acc@thr"], best["auc"]):
                best = dict(result, epoch=round(point, 2))
                patience_counter = 0
                model.save_pretrained(OUTPUT_DIR)
                tokenizer.save_pretrained(OUTPUT_DIR)
                with open(os.path.join(OUTPUT_DIR, "inference_config.json"), "w") as f:
                    json.dump({
                        "max_len": MAX_LEN,
                        "stride": STRIDE,
                        "max_windows": MAX_WINDOWS,
                        "windows_to_average": best["k"],
                        "threshold": best["threshold"],
                        "labels": {"0": "rejected", "1": "accepted"},
                    }, f, indent=2)
                print(f"   -> saved (best so far: dev acc {best['acc@thr']:.4f})")
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print("early stopping")
                    stop = True
                    break
        if stop:
            break

    # ------------------------- final test ----------------------------
    best_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).to(DEVICE)
    test_labels = np.asarray(test_df["label"].tolist())
    diffs, docs, ranks = predict_windows(best_model, test_loader)
    probs = doc_probs(diffs, docs, ranks, len(test_labels), best["k"])
    preds = (probs >= best["threshold"]).astype(int)
    preds_half = (probs >= 0.5).astype(int)

    print("\n" + "=" * 68)
    print("FINAL RESULTS  (copy everything from here)")
    print("=" * 68)
    print(f"best at epoch {best['epoch']} | windows averaged {best['k']} | threshold {best['threshold']:.2f}")
    print(f"dev  : acc {best['acc@thr']:.4f} | macro-F1 {best['macro_f1@thr']:.4f} | AUC {best['auc']:.4f}")
    print(
        f"test : acc {accuracy_score(test_labels, preds):.4f} | "
        f"macro-F1 {f1_score(test_labels, preds, average='macro'):.4f} | "
        f"F1(accepted) {f1_score(test_labels, preds):.4f} | "
        f"AUC {roc_auc_score(test_labels, probs):.4f}"
    )
    print(f"test at plain 0.5 threshold: acc {accuracy_score(test_labels, preds_half):.4f}")
    print("\nconfusion matrix (rows = true, columns = predicted; order rejected, accepted):")
    print(confusion_matrix(test_labels, preds))
    print()
    print(classification_report(test_labels, preds, target_names=["rejected", "accepted"], digits=4))


if __name__ == "__main__":
    train()

device: cuda | mixed precision: True
loading data
   ildc_test.csv: 1517 rows
   ildc_single_dev.csv: 2511 rows
   ildc_multi_dev.csv: 994 rows
   ildc_single_train.csv: 5082 rows
   ildc_multi_train.csv: 32305 rows
removed overlaps: dev judgments also in test 1517 | train judgments also in test 0 | train judgments also in dev 0


config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/534M [00:00<?, ?B/s]

... (16 outputs left out)


epoch 3:  50%|████▉     | 3345/6691 [24:11<24:11,  2.30it/s]


epoch 2.50 | train_loss 0.5053 | 121 min since start
   dev, last 1 window(s): acc@0.5 0.7736 | best threshold 0.36 -> acc 0.7777 | macro-F1 0.7774 | AUC 0.8645
   dev, last 2 window(s): acc@0.5 0.8209 | best threshold 0.48 -> acc 0.8239 | macro-F1 0.8230 | AUC 0.8959
   dev, last 3 window(s): acc@0.5 0.8099 | best threshold 0.33 -> acc 0.8310 | macro-F1 0.8307 | AUC 0.9042
   dev, last 4 window(s): acc@0.5 0.8189 | best threshold 0.49 -> acc 0.8270 | macro-F1 0.8255 | AUC 0.9063
   dev, last 5 window(s): acc@0.5 0.8149 | best threshold 0.44 -> acc 0.8290 | macro-F1 0.8286 | AUC 0.9069
   dev, last 6 window(s): acc@0.5 0.8179 | best threshold 0.45 -> acc 0.8340 | macro-F1 0.8335 | AUC 0.9094
   dev, last 7 window(s): acc@0.5 0.8209 | best threshold 0.35 -> acc 0.8330 | macro-F1 0.8326 | AUC 0.9118
   dev, last 8 window(s): acc@0.5 0.8109 | best threshold 0.38 -> acc 0.8350 | macro-F1 0.8350 | AUC 0.9119
early stopping


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


FINAL RESULTS  (copy everything from here)
best at epoch 1.5 | windows averaged 8 | threshold 0.39
dev  : acc 0.8441 | macro-F1 0.8440 | AUC 0.9126
test : acc 0.8266 | macro-F1 0.8264 | F1(accepted) 0.8332 | AUC 0.9054
test at plain 0.5 threshold: acc 0.7950

confusion matrix (rows = true, columns = predicted; order rejected, accepted):
[[597 158]
 [105 657]]

              precision    recall  f1-score   support

    rejected     0.8504    0.7907    0.8195       755
    accepted     0.8061    0.8622    0.8332       762

    accuracy                         0.8266      1517
   macro avg     0.8283    0.8265    0.8264      1517
weighted avg     0.8282    0.8266    0.8264      1517



In [6]:
!pip install pdfplumber --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 108.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 113.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 103.9 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [7]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pdfplumber

MODEL_DIR = "/kaggle/working/mvp_inlegalbert_best"
CASE_FILE_PATH = "/kaggle/input/datasets/kritiprsd/test-file/test.pdf"
MAX_LEN = 512
HEAD_TOKENS = 128
TAIL_TOKENS = 382
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {DEVICE}")


def extract_text_from_pdf(pdf_path):
    text_parts = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text_parts.append(page_text)
    full_text = "\n".join(text_parts)
    if not full_text.strip():
        raise ValueError(
            "No extractable text found. This PDF is likely scanned/image-based and needs OCR first."
        )
    return full_text


def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
    model.eval()
    return tokenizer, model


def prepare_input(text, tokenizer):
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    budget = HEAD_TOKENS + TAIL_TOKENS
    if len(ids) > budget:
        ids = ids[:HEAD_TOKENS] + ids[-TAIL_TOKENS:]
    ids = [tokenizer.cls_token_id] + ids + [tokenizer.sep_token_id]
    input_ids = torch.full((1, MAX_LEN), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((1, MAX_LEN), dtype=torch.long)
    input_ids[0, :len(ids)] = torch.tensor(ids, dtype=torch.long)
    attention_mask[0, :len(ids)] = 1
    return input_ids.to(DEVICE), attention_mask.to(DEVICE)


def predict(text, tokenizer, model):
    input_ids, attention_mask = prepare_input(text, tokenizer)
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = F.softmax(logits, dim=1).squeeze(0).cpu()
    return {"rejected_prob": probs[0].item(), "accepted_prob": probs[1].item()}


if __name__ == "__main__":
    case_text = extract_text_from_pdf(CASE_FILE_PATH)

    tokenizer, model = load_model()
    result = predict(case_text, tokenizer, model)

    print(f"Chance of being cleared (accepted): {result['accepted_prob']*100:.1f}%")
    print(f"Chance of being rejected: {result['rejected_prob']*100:.1f}%")

using device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Chance of being cleared (accepted): 8.2%
Chance of being rejected: 91.8%


In [8]:
import os

print(os.listdir("mvp_inlegalbert_best"))

['tokenizer_config.json', 'config.json', 'model.safetensors', 'tokenizer.json', 'inference_config.json']


In [9]:
!zip -r mvp_inlegalbert_best.zip mvp_inlegalbert_best

  adding: mvp_inlegalbert_best/ (stored 0%)
  adding: mvp_inlegalbert_best/tokenizer_config.json (deflated 45%)
  adding: mvp_inlegalbert_best/config.json (deflated 53%)
  adding: mvp_inlegalbert_best/model.safetensors (deflated 7%)
  adding: mvp_inlegalbert_best/tokenizer.json (deflated 71%)
  adding: mvp_inlegalbert_best/inference_config.json (deflated 29%)


In [10]:
from IPython.display import FileLink
    
FileLink("mvp_inlegalbert_best.zip")

/kaggle/working/mvp_inlegalbert_best.zip